# Random Forest forecasting: next-day and next-week AQI category

## Why the previous version was not a forecast

The earlier notebook trained on a day's hourly weather and predicted **that same day's** AQI category. That is *same-day estimation*, not forecasting, and it has four problems if the goal is next-day / next-week AQI:

1. **No forecast horizon.** The target was never shifted forward, so nothing was being predicted about the future.
2. **Random station-day split.** Training days sat on both sides of test days, so the model saw the days right before and after each test day. Neighbouring days are strongly autocorrelated, so this inflates scores. A forecast must be tested on a period strictly *after* the training period.
3. **Wrong information set.** The most useful predictors of tomorrow's air quality are today's and recent pollutant levels, but they were excluded as "leakage". For a forecast they are legitimate, because they are known at forecast time. What *is* leakage is any information from the target day itself, such as that day's weather.
4. **`year` as a predictor.** With a time-ordered split the test years are never seen in training, and a Random Forest cannot extrapolate to them.

Random Forest classification is still a reasonable model for AQI *categories*. The fix is in the data design, not the algorithm.

## What this notebook does

- **Unit of prediction:** one row per station-day (about 17,000 rows), not hourly rows with a repeated daily label.
- **Features, all known at the end of day *t*:** daily mean pollutants and weather for day *t*, the previous day, and 3- and 7-day rolling means, the category labels of day *t*, the station, and calendar position of the **target** day (month and day of year, which are known in advance).
- **Targets:** each category in `TARGET_COLUMNS` at day *t+1* (**next day**) and day *t+7* (**next week**, meaning the category 7 days ahead).
- **Split:** chronological. The final 20% of dates form the test set. Training rows whose target date falls on or after the cutoff are purged, so no training label comes from the test period.
- **Baseline:** *persistence* (predict that the future category equals today's). A forecast model is only useful if it beats this, so every result is reported against it.

In [4]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sqlalchemy import create_engine, text

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, classification_report,
        confusion_matrix, f1_score,
    )
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
except ImportError as error:
    raise ImportError(
        "This modeling notebook requires scikit-learn. Install it in this notebook's "
        "Python environment with `%pip install scikit-learn`, then restart the kernel."
    ) from error

# aqi_labels.py holds the China AQI rules, shared with sets.ipynb.
NOTEBOOK_DIR = Path.cwd()
helper_files = list(NOTEBOOK_DIR.rglob("aqi_labels.py"))
if not helper_files:
    raise FileNotFoundError(
        f"aqi_labels.py was not found under {NOTEBOOK_DIR}. "
        "Copy the shared aqi_labels.py file into this folder or one of its subfolders."
    )

helper_dir = helper_files[0].parent.resolve()
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from aqi_labels import (  # noqa: E402
    POLLUTANTS,
    TARGET_COLUMNS,
    build_daily_labels,
    require_columns,
)

RANDOM_STATE = 42
TEST_FRACTION = 0.20                      # final 20% of dates are the test period
HORIZONS = {"next_day": 1, "next_week": 7}  # days ahead of day t
SHOW_DETAILS = True                       # confusion matrices / per-class reports
SOURCE_TABLE = "all_regions"
EXPECTED_SOURCE_ROWS = 418_753
DB_URL = os.environ.get("AIR_QUALITY_DB_URL", "mysql+pymysql://root:@localhost/air_quality")

## 1. Load the cleaned `all_regions` table

In [5]:
if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load {SOURCE_TABLE!r} from the air_quality database. Check MySQL, "
        "AIR_QUALITY_DB_URL, the table name, SQLAlchemy, and PyMySQL. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded source rows: {len(source_df):,}")
if len(source_df) != EXPECTED_SOURCE_ROWS:
    print(
        f"Note: expected {EXPECTED_SOURCE_ROWS:,} rows, but found {len(source_df):,}. "
        "Continuing after validating the required columns."
    )
require_columns(source_df, f"air_quality.{SOURCE_TABLE}")
display(source_df.head())

Loaded source rows: 420,768
Note: expected 418,753 rows, but found 420,768. Continuing after validating the required columns.


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


## 2. Daily China AQI and pollutant category labels

Same shared `build_daily_labels` as `sets.ipynb`, so the breakpoints, validity gates and category boundaries are unchanged. The output is a station-day table. These labels are used as **future targets** and as "today's category" features, never as same-day targets for same-day features.

In [ ]:
# The shared label function uses this field to track upstream interpolation provenance.
# If the database table omitted it, add an explicit "None" marker so label calculation
# can proceed. This does NOT prove that every reading was measured; any earlier
# interpolation is simply unidentifiable from this table and is counted as available data.
source_for_labels = source_df.copy()
if "interpolated_columns" not in source_for_labels.columns:
    print(
        "Warning: 'interpolated_columns' is missing. AQI labels will still be calculated, "
        "but the notebook cannot distinguish interpolated readings from measured ones."
    )
    source_for_labels["interpolated_columns"] = "None"

daily_labels, _ = build_daily_labels(source_for_labels)

print(f"Daily station records: {len(daily_labels):,}")
display(daily_labels[TARGET_COLUMNS].notna().sum().rename("valid_station_days").to_frame())


## 3. Build one row per station-day with history features

Hourly readings are aggregated to daily means (rain is summed). Each station is then placed on a **complete daily calendar**, so a missing day stays a gap instead of silently shifting "yesterday" or "next week" to the wrong date. Every feature uses only day *t* and earlier.

In [ ]:
# --- hourly -> station-day -------------------------------------------------
work = source_df.copy()
for column in ("year", "month", "day"):
    work[column] = pd.to_numeric(work[column], errors="coerce")
work["date"] = pd.to_datetime(work[["year", "month", "day"]], errors="coerce").dt.normalize()
work = work.dropna(subset=["date", "station"])

POLLUTANT_COLUMNS = [c for c in POLLUTANTS if c in work.columns]
WEATHER_COLUMNS = [c for c in ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"] if c in work.columns]
if not POLLUTANT_COLUMNS:
    raise KeyError("No pollutant columns from aqi_labels.POLLUTANTS were found in the source table.")
if not WEATHER_COLUMNS:
    raise KeyError("No expected weather columns were found (TEMP, PRES, DEWP, RAIN, WSPM).")
MEASURED_COLUMNS = POLLUTANT_COLUMNS + WEATHER_COLUMNS
for column in MEASURED_COLUMNS:
    work[column] = pd.to_numeric(work[column], errors="coerce")

aggregations = {column: "mean" for column in MEASURED_COLUMNS}
if "RAIN" in aggregations:
    aggregations["RAIN"] = lambda s: s.sum(min_count=1)
daily = work.groupby(["station", "date"], as_index=False).agg(aggregations)

# --- attach the daily category labels (match on station + date) -------------
label_table = daily_labels.copy()
if "station" not in label_table.columns:
    raise KeyError(f"daily_labels has no 'station' column. Columns: {list(label_table.columns)}")
if "date" in label_table.columns:
    label_table["date"] = pd.to_datetime(label_table["date"], errors="coerce").dt.normalize()
elif {"year", "month", "day"} <= set(label_table.columns):
    label_table["date"] = pd.to_datetime(
        label_table[["year", "month", "day"]].apply(pd.to_numeric, errors="coerce"), errors="coerce"
    ).dt.normalize()
else:
    raise KeyError(
        "daily_labels needs a 'date' column or year/month/day columns to join on. "
        f"Columns: {list(label_table.columns)}"
    )
if label_table.duplicated(["station", "date"]).any():
    raise ValueError("daily_labels has more than one row for some station-day.")
label_table = label_table[["station", "date", *TARGET_COLUMNS]]

daily = daily.merge(label_table, on=["station", "date"], how="left")
for target in TARGET_COLUMNS:  # plain object column with NaN for missing, safe to shift/impute
    daily[target] = daily[target].astype(object).where(daily[target].notna(), np.nan)

# --- complete daily calendar per station -----------------------------------
def complete_calendar(frame):
    parts = []
    for station, group in frame.groupby("station"):
        group = group.set_index("date").sort_index()
        full_range = pd.date_range(group.index.min(), group.index.max(), freq="D")
        group = group.reindex(full_range)
        group["station"] = station
        group.index.name = "date"
        parts.append(group.reset_index())
    return pd.concat(parts, ignore_index=True)

daily = complete_calendar(daily).sort_values(["station", "date"]).reset_index(drop=True)

# --- history features: today, yesterday, 3- and 7-day means (all <= day t) --
by_station = daily.groupby("station")
history = {}
for column in MEASURED_COLUMNS:
    history[f"{column}_d0"] = daily[column]
    history[f"{column}_lag1"] = by_station[column].shift(1)
    history[f"{column}_roll3"] = by_station[column].transform(lambda s: s.rolling(3, min_periods=1).mean())
    history[f"{column}_roll7"] = by_station[column].transform(lambda s: s.rolling(7, min_periods=1).mean())
history = pd.DataFrame(history, index=daily.index)

today_labels = daily[TARGET_COLUMNS].add_prefix("today_")   # category known at end of day t
feature_base = pd.concat([daily[["station"]], history, today_labels], axis=1)

HISTORY_FEATURES = list(history.columns)
TODAY_FEATURES = list(today_labels.columns)
print(f"Station-days (complete calendar): {len(daily):,}")
print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()}")
print(f"History features: {len(HISTORY_FEATURES)} numeric + {len(TODAY_FEATURES)} 'today' categories + station")

## 4. Chronological split with purging

Dates are ordered and the last `TEST_FRACTION` of them become the test period. For a horizon of *h* days:

- **Test rows:** forecast origin *t* on or after the cutoff.
- **Training rows:** target date *t+h* strictly **before** the cutoff, so even the training *labels* never come from the test period.

Random shuffling is not used anywhere.

In [ ]:
all_dates = np.sort(daily["date"].unique())
cutoff = pd.Timestamp(all_dates[int(len(all_dates) * (1 - TEST_FRACTION))])
print(f"Training origins/targets before {cutoff.date()}  |  test origins from {cutoff.date()} "
      f"to {pd.Timestamp(all_dates[-1]).date()}")


def calendar_features(dates):
    # Calendar position of the TARGET day: known in advance, so not leakage.
    doy = dates.dt.dayofyear
    month = dates.dt.month
    return pd.DataFrame({
        "target_month_sin": np.sin(2 * np.pi * (month - 1) / 12),
        "target_month_cos": np.cos(2 * np.pi * (month - 1) / 12),
        "target_doy_sin": np.sin(2 * np.pi * (doy - 1) / 365.25),
        "target_doy_cos": np.cos(2 * np.pi * (doy - 1) / 365.25),
    }, index=dates.index)


CALENDAR_FEATURES = list(calendar_features(daily["date"].head(5)).columns)
NUMERIC_FEATURES = HISTORY_FEATURES + CALENDAR_FEATURES
CATEGORICAL_FEATURES = ["station"] + TODAY_FEATURES
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

## 5. Train and evaluate: Random Forest vs. persistence baseline

One classifier per (horizon, target). Preprocessing is fit on training rows only. Each model is scored next to the persistence baseline on **exactly the same test rows** (rows where today's category and the future category both exist).

In [ ]:
forecast_models = {}
rows = []

for horizon_name, horizon_days in HORIZONS.items():
    target_date = daily["date"] + pd.Timedelta(days=horizon_days)
    X_all = pd.concat([feature_base, calendar_features(target_date)], axis=1)[FEATURE_COLUMNS]

    for target in TARGET_COLUMNS:
        y_future = daily.groupby("station")[target].shift(-horizon_days)
        today = daily[target]

        train_mask = y_future.notna() & (target_date < cutoff)
        test_mask = y_future.notna() & today.notna() & (daily["date"] >= cutoff)
        if train_mask.sum() == 0 or test_mask.sum() == 0:
            print(f"Skipping {horizon_name}/{target}: no labeled train or test rows.")
            continue

        y_train = y_future[train_mask].astype(str)
        y_test = y_future[test_mask].astype(str)
        if y_train.nunique() < 2:
            print(f"Skipping {horizon_name}/{target}: fewer than two classes in training.")
            continue
        X_train, X_test = X_all[train_mask], X_all[test_mask]
        baseline = today[test_mask].astype(str)  # persistence: "it will be like today"

        preprocessing = ColumnTransformer([
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NUMERIC_FEATURES),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), CATEGORICAL_FEATURES),
        ])
        model = Pipeline([
            ("preprocessing", preprocessing),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        forecast_models[(horizon_name, target)] = model

        row = {
            "horizon": horizon_name,
            "target": target,
            "train_rows": int(train_mask.sum()),
            "test_rows": int(test_mask.sum()),
            "rf_accuracy": accuracy_score(y_test, predictions),
            "persistence_accuracy": accuracy_score(y_test, baseline),
            "rf_balanced_acc": balanced_accuracy_score(y_test, predictions),
            "persistence_balanced_acc": balanced_accuracy_score(y_test, baseline),
            "rf_macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
            "persistence_macro_f1": f1_score(y_test, baseline, average="macro", zero_division=0),
        }
        row["accuracy_gain_vs_persistence"] = row["rf_accuracy"] - row["persistence_accuracy"]
        rows.append(row)

        if SHOW_DETAILS:
            labels = sorted(set(y_train) | set(y_test) | set(predictions))
            print(f"\n=== {horizon_name} (t+{horizon_days}d) | {target} ===")
            print(f"Train rows: {row['train_rows']:,} | Test rows: {row['test_rows']:,}")
            print(f"RF accuracy {row['rf_accuracy']:.3f} vs persistence {row['persistence_accuracy']:.3f}")
            display(pd.DataFrame(
                confusion_matrix(y_test, predictions, labels=labels),
                index=pd.Index(labels, name="actual"),
                columns=pd.Index(labels, name="predicted"),
            ))
            display(pd.DataFrame(classification_report(
                y_test, predictions, labels=labels, output_dict=True, zero_division=0
            )).T)
            names = model.named_steps["preprocessing"].get_feature_names_out()
            importances = model.named_steps["classifier"].feature_importances_
            display(pd.Series(importances, index=names).sort_values(ascending=False)
                    .head(10).rename("importance").to_frame())

forecast_results = pd.DataFrame(rows).set_index(["horizon", "target"])
display(forecast_results.round(3))

## How to read the results

- **Compare with persistence, not with 100%.** `accuracy_gain_vs_persistence` above zero means the Random Forest adds something beyond "tomorrow looks like today". Near zero or negative means it does not.
- **Expect skill to fall from next-day to next-week.** Air quality is only weakly predictable a week out without weather forecasts, so next-week scores close to persistence or climatology are normal, not a bug.
- **No weather of the target day is used.** Real operational forecasts would feed in a weather forecast for day *t+h*. Adding that is a natural next step, and it must be the *forecast* available at time *t*, not the observed weather.
- **Test period is one contiguous block.** It covers only part of a seasonal cycle, so treat the numbers as indicative. A rolling-origin check (`TimeSeriesSplit`) over several cutoffs would be a more robust estimate.
- **Label provenance caveat from the earlier notebook still applies:** some hourly values behind daily labels and features were filled by short linear interpolation (see `sets.ipynb`).
- **"Next week" here means the category 7 days ahead.** If you instead want the category of the *mean AQI over the next 7 days*, only the target construction in section 5 changes.